In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = "dataset/FAOSTAT_data_en_11-1-2024.csv"
OUTPUT_FILE = "Thailand_Temperature_Clean.csv"



In [2]:

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(
    INPUT_FILE,
    encoding="latin1"
)

print("=" * 70)
print("ORIGINAL DATA")
print("=" * 70)

print("Shape:", df.shape)
print(df.columns.tolist())


ORIGINAL DATA
Shape: (241893, 14)
['ï»¿Domain Code', 'Domain', 'Area Code (M49)', 'Area', 'Element Code', 'Element', 'Months Code', 'Months', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Flag Description']


In [3]:

# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

print("\nColumns:")
print(df.columns.tolist())



Columns:
['ï»¿Domain Code', 'Domain', 'Area Code (M49)', 'Area', 'Element Code', 'Element', 'Months Code', 'Months', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Flag Description']


In [4]:

# ============================================================
# 3. SELECT THAILAND
# ============================================================

df = df[
    df["Area"].astype(str).str.strip() == "Thailand"
].copy()

print("\nThailand rows:", len(df))


Thailand rows: 1071


In [5]:

# ============================================================
# 4. SELECT TEMPERATURE CHANGE
# ============================================================

df = df[
    df["Element"].astype(str).str.strip()
    == "Temperature change"
].copy()


In [6]:

# ============================================================
# 5. SELECT ANNUAL DATA
# ============================================================

df = df[
    df["Months"].astype(str).str.strip()
    == "Meteorological year"
].copy()

In [7]:

# ============================================================
# 6. SELECT ONLY REQUIRED COLUMNS
# ============================================================

df = df[
    [
        "Area",
        "Year",
        "Unit",
        "Value",
        "Flag"
    ]
].copy()

In [8]:

# ============================================================
# 7. CONVERT DATA TYPE
# ============================================================

df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
)

df["Value"] = pd.to_numeric(
    df["Value"],
    errors="coerce"
)


In [9]:

# ============================================================
# 8. REMOVE MISSING VALUES
# ============================================================

print("\nMissing values:")
print(df.isnull().sum())

df = df.dropna(
    subset=["Year", "Value"]
)


Missing values:
Area     0
Year     0
Unit     0
Value    0
Flag     0
dtype: int64


In [10]:

# ============================================================
# 9. REMOVE DUPLICATES
# ============================================================

duplicates = df.duplicated().sum()

print("\nDuplicate rows:", duplicates)

df = df.drop_duplicates()




Duplicate rows: 0


In [11]:
# ============================================================
# 10. SORT BY YEAR
# ============================================================

df = df.sort_values(
    "Year"
).reset_index(drop=True)



In [12]:

# ============================================================
# 11. CHECK DUPLICATE YEARS
# ============================================================

duplicate_years = (
    df["Year"].duplicated().sum()
)

print(
    "Duplicate years:",
    duplicate_years
)


Duplicate years: 0


In [13]:
# ============================================================
# 12. CHECK MISSING YEARS
# ============================================================

min_year = int(df["Year"].min())
max_year = int(df["Year"].max())

expected_years = set(
    range(min_year, max_year + 1)
)

actual_years = set(
    df["Year"].astype(int)
)

missing_years = sorted(
    expected_years - actual_years
)

print("\nYear range:")
print(min_year, "-", max_year)

print(
    "Missing years:",
    missing_years
)


Year range:
1961 - 2023
Missing years: []


In [14]:

# ============================================================
# 13. IF MISSING YEAR → INTERPOLATE
# ============================================================

if len(missing_years) > 0:

    full_years = pd.DataFrame({
        "Year": range(
            min_year,
            max_year + 1
        )
    })

    df = full_years.merge(
        df,
        on="Year",
        how="left"
    )

    df["Value"] = (
        df["Value"]
        .interpolate(
            method="linear"
        )
    )

In [15]:
# ============================================================
# 14. FINAL DATA CLEAN
# ============================================================

df = df[
    ["Year", "Value"]
].copy()

df = df.sort_values(
    "Year"
).reset_index(drop=True)


In [ ]:

# ============================================================
# 15. CHECK FINAL DATA
# ============================================================

print("\n" + "=" * 70)
print("CLEAN DATA")
print("=" * 70)

print(df.to_string(index=False))


print("\nStatistics:")
print(df["Value"].describe())